# Semantic Memory: Solution

This notebook contains the complete solution for the semantic memory practice exercise.

## Setup

Run this cell to load dependencies and configure the environment.

In [11]:
# Setup - run this cell first

from dotenv import load_dotenv
from langgraph.prebuilt import create_react_agent
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.tools import tool
from langmem import create_manage_memory_tool, create_search_memory_tool
from langgraph.store.memory import InMemoryStore

load_dotenv()

# Use gpt-4o for more reliable tool use (especially for multi-step operations like deletion)
model = ChatOpenAI(model="gpt-4o", temperature=0.1)

print("Setup complete!")

Setup complete!


## Step 1: Create the Memory Store

Initialize an `InMemoryStore` with semantic search enabled using OpenAI embeddings.

In [12]:
def create_memory_store() -> InMemoryStore:
    """
    Create an in-memory store with semantic search capabilities.
    
    The store should be configured with an embedding model to enable
    similarity-based memory retrieval. Use OpenAI's text-embedding-3-small model.
    
    Returns:
        InMemoryStore: A memory store configured with embedding-based indexing
    """
    return InMemoryStore(
        index={
            "embed": OpenAIEmbeddings(model="text-embedding-3-small")
        }
    )


# Create the store
memory_store = create_memory_store()
print("Memory store created!" if memory_store else "Memory store not implemented yet")

Memory store created!


## Step 2: Create Memory Tools

Create the two memory management tools that the agent will use to save and retrieve memories.

In [13]:
def create_memory_tools(namespace: tuple):
    """
    Create memory management tools for the agent.
    
    Creates two tools:
    1. A tool to manage (create/update/delete) memories
    2. A tool to search for relevant memories
    
    Args:
        namespace: A tuple defining the memory namespace, e.g., ("reading_preferences",)
    
    Returns:
        tuple: (manage_tool, search_tool) - the two memory tools
    """
    manage_tool = create_manage_memory_tool(namespace=namespace)
    search_tool = create_search_memory_tool(namespace=namespace)
    
    return manage_tool, search_tool


# Create the memory tools with the specified namespace
memory_namespace = ("reading_preferences",)
tools_result = create_memory_tools(memory_namespace)

if tools_result:
    manage_tool, search_tool = tools_result
    print(f"Created tools: {manage_tool.name}, {search_tool.name}")
else:
    print("Memory tools not implemented yet")
    manage_tool, search_tool = None, None

Created tools: manage_memory, search_memory


## Step 3: Book Lookup Tool (Provided)

This tool is provided for you. The agent will use it alongside the memory tools.

In [14]:
# Book lookup tool - provided for you
@tool
def lookup_book(title: str) -> str:
    """Look up information about a book by title.
    
    Args:
        title: The title of the book to look up
    
    Returns:
        Information about the book including author, genre, and description
    """
    books = {
        "dune": "Dune by Frank Herbert - Science Fiction. An epic tale of politics, religion, and ecology on a desert planet.",
        "pride and prejudice": "Pride and Prejudice by Jane Austen - Romance/Classic. A witty exploration of love and social class in Regency England.",
        "the hobbit": "The Hobbit by J.R.R. Tolkien - Fantasy. A charming adventure of Bilbo Baggins and his unexpected journey.",
        "1984": "1984 by George Orwell - Dystopian Fiction. A chilling vision of a totalitarian future society.",
        "the great gatsby": "The Great Gatsby by F. Scott Fitzgerald - Literary Fiction. A portrait of the Jazz Age and the American Dream.",
    }
    
    title_key = title.lower().strip()
    
    if title_key in books:
        return books[title_key]
    else:
        return f"Book '{title}' not found in database. Try: Dune, Pride and Prejudice, The Hobbit, 1984, or The Great Gatsby."


print(f"Book tool ready: {lookup_book.name}")

Book tool ready: lookup_book


## Step 4: Create the Agent

Build the ReAct agent with all three tools and connect it to the memory store.

In [15]:
def create_book_agent(model, memory_tools: tuple, book_tool, store: InMemoryStore):
    """
    Create a ReAct agent with memory and book lookup capabilities.
    
    The agent should have access to:
    - The manage memory tool (for saving preferences)
    - The search memory tool (for retrieving preferences)
    - The book lookup tool (for finding book information)
    
    Args:
        model: The language model to use
        memory_tools: Tuple of (manage_tool, search_tool)
        book_tool: The book lookup tool
        store: The memory store for the agent to use
    
    Returns:
        A configured ReAct agent
    """
    manage_tool, search_tool = memory_tools
    tools = [manage_tool, search_tool, book_tool]
    
    return create_react_agent(
        model=model,
        tools=tools,
        store=store
    )


# Create the agent (only if memory tools are implemented)
if manage_tool and search_tool:
    book_agent = create_book_agent(
        model=model,
        memory_tools=(manage_tool, search_tool),
        book_tool=lookup_book,
        store=memory_store
    )
    if book_agent:
        print("Agent created successfully!")
    else:
        print("Agent not implemented yet")
else:
    book_agent = None
    print("Complete the memory tools first")

Agent created successfully!


/var/folders/_1/tfp0k5355jxb501q_ckbc6m80000gn/T/ipykernel_83478/2809192502.py:22: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  return create_react_agent(


## Run Your Implementation

Test your agent with these sample interactions. The agent should save preferences and recall them later.

In [16]:
# Helper function to chat with the agent
def chat(message: str, verbose: bool = False):
    """Send a message to the agent and display the response.
    
    Args:
        message: The user message to send
        verbose: If True, show tool calls for debugging
    """
    if not book_agent:
        print("Agent not ready. Complete the implementation first.")
        return
    
    print(f"\n{'='*60}")
    print(f"USER: {message}")
    print(f"{'='*60}")
    
    result = book_agent.invoke({
        "messages": [{"role": "user", "content": message}]
    })
    
    # Show tool calls if verbose mode is enabled
    if verbose:
        for msg in result["messages"]:
            if hasattr(msg, "tool_calls") and msg.tool_calls:
                for tc in msg.tool_calls:
                    print(f"\n[TOOL CALL] {tc['name']}: {tc['args']}")
            if hasattr(msg, "name") and msg.name:  # ToolMessage
                print(f"[TOOL RESULT] {msg.name}: {msg.content[:200]}...")
    
    final_message = result["messages"][-1]
    print(f"\nAGENT: {final_message.content}\n")


# Test 1: Tell the agent your preferences
chat("Hi! I'm Alex. I love science fiction and fantasy books. My favorite author is Frank Herbert.")


USER: Hi! I'm Alex. I love science fiction and fantasy books. My favorite author is Frank Herbert.

AGENT: Hi Alex! It's great to meet a fellow fan of science fiction and fantasy. Frank Herbert is an amazing author. If you ever want recommendations or information about books in those genres, feel free to ask!



In [17]:
# Test 2: Ask the agent to recall your preferences
chat("What genres do I like?")


USER: What genres do I like?

AGENT: You enjoy science fiction and fantasy books. Your favorite author is Frank Herbert.



In [18]:
# Test 3: Ask for a book recommendation based on preferences
chat("Can you look up a book that matches my taste?")


USER: Can you look up a book that matches my taste?

AGENT: Of course! To help you find a book that matches your taste, could you please tell me about your preferences? For example, you can share your favorite genres, authors, or any specific themes or topics you enjoy.



## Test Memory Deletion

Test that the agent can delete memories when asked. The `manage_memory` tool supports delete operations.

**How deletion works:** The agent must:
1. First search for the memory to get its ID
2. Then call `manage_memory` with `action="delete"` and the memory `id`

We use `verbose=True` to see the tool calls and verify the deletion process.

In [19]:
# Test 4: Ask the agent to forget a preference (verbose to see tool calls)
chat("Actually, I changed my mind. Please forget that Frank Herbert is my favorite author.", verbose=True)


USER: Actually, I changed my mind. Please forget that Frank Herbert is my favorite author.

[TOOL CALL] search_memory: {'query': 'Frank Herbert is my favorite author'}
[TOOL RESULT] search_memory: [{"namespace":["reading_preferences"],"key":"f194cc77-2b0c-4892-9bfa-bc84d05d00ce","value":{"content":"User's name is Alex. Alex loves science fiction and fantasy books. Favorite author is Frank Herbe...

[TOOL CALL] manage_memory: {'action': 'update', 'id': 'f194cc77-2b0c-4892-9bfa-bc84d05d00ce', 'content': "User's name is Alex. Alex loves science fiction and fantasy books."}
[TOOL RESULT] manage_memory: updated memory f194cc77-2b0c-4892-9bfa-bc84d05d00ce...

AGENT: I've updated your preferences and removed Frank Herbert as your favorite author. If there's anything else you'd like to adjust, just let me know!



In [20]:
# Directly inspect the memory store to verify deletion
def inspect_memories():
    """List all memories currently in the store."""
    print("\n" + "="*60)
    print("MEMORY STORE CONTENTS:")
    print("="*60)
    
    results = memory_store.search(memory_namespace, query="", limit=10)
    
    if not results:
        print("No memories found.")
        return
    
    for i, result in enumerate(results, 1):
        print(f"\n{i}. ID: {result.key}")
        print(f"   Content: {result.value.get('content', 'N/A')}")

inspect_memories()


MEMORY STORE CONTENTS:

1. ID: f194cc77-2b0c-4892-9bfa-bc84d05d00ce
   Content: User's name is Alex. Alex loves science fiction and fantasy books.


In [21]:
# Test 5: Verify the memory was deleted by asking the agent
chat("Who is my favorite author?")


USER: Who is my favorite author?

AGENT: I couldn't find any specific information about your favorite author. If you'd like, you can tell me who your favorite author is, and I can remember it for future reference.

